# Regime Filter Deep Dive

**Finding:** Regime filters didn't improve beat rate.

**Question:** Why? Let's look at trade-by-trade performance.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

# Load data
DATA_DIR = Path("../data/raw")

price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")
mvrv = pd.read_parquet(DATA_DIR / "mvrv.parquet").rename(columns={"value": "mvrv"}).set_index("time")
sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
realized_loss = pd.read_parquet(DATA_DIR / "realized_loss.parquet").rename(columns={"value": "realized_loss"}).set_index("time")

df = price.join(mvrv, how='inner').join(sopr, how='inner').join(sopr_sth, how='inner').join(realized_loss, how='inner')
df = df.sort_index()

df['ma200'] = df['price'].rolling(200).mean()
df['rl_ma30'] = df['realized_loss'].rolling(30).mean()
df['rl_std30'] = df['realized_loss'].rolling(30).std()
df['rl_zscore'] = (df['realized_loss'] - df['rl_ma30']) / df['rl_std30']
df['regime_bull'] = df['price'] > df['ma200']

df_test = df[df.index >= '2018-12-15'].copy().dropna()
print(f"Data: {len(df_test)} rows")

In [ ]:
def sopr_rl_entry(df, rl_z_threshold=0.5):
    sopr_signal = (df['sopr'] < 1) & (df['sopr_sth'] < 1)
    rl_signal = df['rl_zscore'] > rl_z_threshold
    combined = sopr_signal & rl_signal
    entries = combined & ~combined.shift(1).fillna(False)
    return entries

def backtest_with_details(df, entries, mvrv_trigger=2.0, trailing_pct=0.25, stop_loss=0.20, max_hold_days=365):
    trades = []
    entry_indices = entries[entries].index.tolist()
    close = df['price']
    
    i = 0
    while i < len(entry_indices):
        entry_date = entry_indices[i]
        entry_idx = df.index.get_loc(entry_date)
        entry_price = close.iloc[entry_idx]
        
        peak_price = entry_price
        max_drawdown = 0
        trailing_active = False
        
        exit_date = None
        exit_price = None
        exit_reason = None
        
        for j in range(entry_idx + 1, len(df)):
            current_date = df.index[j]
            current_price = close.iloc[j]
            current_mvrv = df['mvrv'].iloc[j]
            days_held = j - entry_idx
            
            if current_price > peak_price:
                peak_price = current_price
            
            current_dd = (current_price - peak_price) / peak_price
            if current_dd < max_drawdown:
                max_drawdown = current_dd
            
            pnl = (current_price - entry_price) / entry_price
            
            if not trailing_active and current_mvrv >= mvrv_trigger:
                trailing_active = True
            
            if trailing_active:
                trail_stop = peak_price * (1 - trailing_pct)
                if current_price <= trail_stop:
                    exit_date = current_date
                    exit_price = trail_stop
                    exit_reason = 'mvrv_trail'
                    break
            
            if not trailing_active and stop_loss and pnl <= -stop_loss:
                exit_date = current_date
                exit_price = entry_price * (1 - stop_loss)
                exit_reason = 'stop_loss'
                break
            
            if days_held >= max_hold_days:
                exit_date = current_date
                exit_price = current_price
                exit_reason = 'max_hold'
                break
        
        if exit_date is None:
            exit_date = df.index[-1]
            exit_price = close.iloc[-1]
            exit_reason = 'end_of_data'
        
        pnl = (exit_price - entry_price) / entry_price
        max_gain = (peak_price - entry_price) / entry_price
        
        trades.append({
            'entry_date': entry_date,
            'exit_date': exit_date,
            'entry_price': entry_price,
            'exit_price': exit_price,
            'pnl_pct': pnl,
            'max_gain': max_gain,
            'max_drawdown': max_drawdown,
            'days_held': (exit_date - entry_date).days,
            'exit_reason': exit_reason,
            'regime_at_entry': 'BULL' if df.loc[entry_date, 'regime_bull'] else 'BEAR',
            'mvrv_at_entry': df.loc[entry_date, 'mvrv']
        })
        
        while i < len(entry_indices) and entry_indices[i] <= exit_date:
            i += 1
    
    return pd.DataFrame(trades)

In [ ]:
# Run backtest
entries = sopr_rl_entry(df_test, 0.5)
trades = backtest_with_details(df_test, entries)

print("ALL TRADES - DETAILED VIEW")
print("="*140)
print(f"{'Entry':<12} {'Exit':<12} {'Entry $':>10} {'Exit $':>10} {'PnL':>8} {'Max Gain':>10} {'Max DD':>10} {'Days':>6} {'Exit':>12} {'Regime':>8} {'MVRV':>6}")
print("-"*140)

for _, t in trades.iterrows():
    print(f"{str(t['entry_date'].date()):<12} {str(t['exit_date'].date()):<12} "
          f"{t['entry_price']:>10,.0f} {t['exit_price']:>10,.0f} "
          f"{t['pnl_pct']*100:>+7.0f}% {t['max_gain']*100:>+9.0f}% {t['max_drawdown']*100:>+9.0f}% "
          f"{t['days_held']:>6} {t['exit_reason']:>12} {t['regime_at_entry']:>8} {t['mvrv_at_entry']:>6.2f}")

In [ ]:
# Performance by regime
print("\n" + "="*80)
print("PERFORMANCE BY REGIME AT ENTRY")
print("="*80)

for regime in ['BULL', 'BEAR']:
    regime_trades = trades[trades['regime_at_entry'] == regime]
    if len(regime_trades) == 0:
        print(f"\n{regime}: No trades")
        continue
    
    print(f"\n{regime} MARKET ENTRIES ({len(regime_trades)} trades):")
    print(f"  Win Rate: {(regime_trades['pnl_pct'] > 0).mean()*100:.0f}%")
    print(f"  Avg PnL: {regime_trades['pnl_pct'].mean()*100:+.1f}%")
    print(f"  Total Return: {((1+regime_trades['pnl_pct']).prod()-1)*100:+.0f}%")
    print(f"  Avg Max Gain: {regime_trades['max_gain'].mean()*100:+.0f}%")
    print(f"  Avg Max DD: {regime_trades['max_drawdown'].mean()*100:.0f}%")
    
    print(f"  Exit reasons:")
    for reason in regime_trades['exit_reason'].value_counts().items():
        print(f"    {reason[0]}: {reason[1]}")

In [ ]:
# Key insight visualization
fig = go.Figure()

bull_trades = trades[trades['regime_at_entry'] == 'BULL']
bear_trades = trades[trades['regime_at_entry'] == 'BEAR']

fig.add_trace(go.Bar(
    name='Bull Entries',
    x=['Count', 'Win Rate %', 'Avg PnL %'],
    y=[len(bull_trades), 
       (bull_trades['pnl_pct'] > 0).mean()*100 if len(bull_trades) > 0 else 0,
       bull_trades['pnl_pct'].mean()*100 if len(bull_trades) > 0 else 0],
    marker_color='green'
))

fig.add_trace(go.Bar(
    name='Bear Entries',
    x=['Count', 'Win Rate %', 'Avg PnL %'],
    y=[len(bear_trades),
       (bear_trades['pnl_pct'] > 0).mean()*100 if len(bear_trades) > 0 else 0,
       bear_trades['pnl_pct'].mean()*100 if len(bear_trades) > 0 else 0],
    marker_color='red'
))

fig.update_layout(barmode='group', title='Bull vs Bear Entry Performance', height=400)
fig.show()

In [ ]:
# The key question: Would filtering out bear entries help or hurt?
print("\n" + "="*80)
print("WHAT IF WE FILTERED OUT BEAR MARKET ENTRIES?")
print("="*80)

if len(bull_trades) > 0:
    bull_total = ((1 + bull_trades['pnl_pct']).prod() - 1) * 100
else:
    bull_total = 0

if len(bear_trades) > 0:
    bear_total = ((1 + bear_trades['pnl_pct']).prod() - 1) * 100
else:
    bear_total = 0

all_total = ((1 + trades['pnl_pct']).prod() - 1) * 100

print(f"\nTotal return WITH bear entries: {all_total:+.0f}%")
print(f"Total return WITHOUT bear entries (bull only): {bull_total:+.0f}%")
print(f"\nBear entries contributed: {bear_total:+.0f}%")

if bear_total > 0:
    print(f"\n💡 INSIGHT: Bear market entries were PROFITABLE!")
    print(f"   Filtering them out would HURT the strategy.")
elif bear_total < -10:
    print(f"\n💡 INSIGHT: Bear market entries lost significant money.")
    print(f"   But walk-forward still didn't improve - maybe timing issue?")
else:
    print(f"\n💡 INSIGHT: Bear market entries roughly break-even.")
    print(f"   Stop loss is protecting us effectively.")

In [ ]:
# Final summary
print("\n" + "="*80)
print("CONCLUSION")
print("="*80)

print("""
The regime filter doesn't help because:

1. CONTRARIAN signals are SUPPOSED to fire in bear markets
   - SOPR < 1 = people selling at loss = happens in downtrends
   - The signal IS the fear/capitulation detection

2. The stop-loss already protects us
   - 20% stop limits downside on bad entries
   - Bear market trades that don't work out get stopped

3. Some of the BEST entries happen in bear markets
   - March 2020 COVID crash
   - Late 2022 FTX bottom
   - These are exactly what the signal is designed to catch!

UPDATED PRINCIPLE:
"Regime filters help TREND-FOLLOWING strategies.
 CONTRARIAN strategies don't need them - their built-in
 stop-loss provides sufficient protection."
""")